# Wire Geometry Generation

Bond wires in semiconductor packages follow a **loop shape** from a die pad to a
lead frame pad.  We model each wire as a cubic Bezier curve with four control
points encoding start, end, loop height, and curvature asymmetry.

## BondWireProfile

Each wire is described by a `BondWireProfile` dataclass:

- `start_xy` / `end_xy` — pad positions
- `loop_height` — peak height above the baseline
- `diameter` — wire thickness in pixels
- `curvature` — asymmetry factor [-1, 1]
- `material` — gold, copper, or aluminum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from udm_epic6.models.wire_geometry import (
    BondWireProfile, generate_wire_profile, render_wire_mask,
    _bezier_control_points, _evaluate_bezier,
)

In [ ]:
# Generate random wire profiles
rng = np.random.default_rng(42)
profiles = generate_wire_profile(rng, image_size=(256, 256), n_wires=4)

for i, p in enumerate(profiles):
    print(f"Wire {i}: start={p.start_xy}, end={p.end_xy}, "
          f"height={p.loop_height:.1f}, diam={p.diameter:.1f}, "
          f"material={p.material}")

In [ ]:
# Visualise Bezier curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot wire paths
ax = axes[0]
for i, p in enumerate(profiles):
    cps = _bezier_control_points(p)
    curve = _evaluate_bezier(cps, n_samples=200)
    ax.plot(curve[:, 0], curve[:, 1], linewidth=p.diameter * 0.5, label=f"Wire {i} ({p.material})")
    ax.plot(*p.start_xy, 'gs', markersize=6)
    ax.plot(*p.end_xy, 'rs', markersize=6)
ax.set_xlim(0, 256)
ax.set_ylim(256, 0)
ax.set_title("Wire Bezier Paths")
ax.legend(fontsize=8)
ax.set_aspect('equal')

# Plot rendered masks
ax = axes[1]
combined_mask = np.zeros((256, 256), dtype=np.float32)
for p in profiles:
    combined_mask += render_wire_mask(p, 256, 256).astype(np.float32) / 255.0
ax.imshow(np.clip(combined_mask, 0, 1), cmap='gray')
ax.set_title("Combined Wire Mask")

fig.tight_layout()
plt.show()

## Curvature parameter

The `curvature` parameter controls asymmetry of the loop:
- `0.0` = symmetric loop (peak at midpoint)
- `> 0` = peak shifted toward start
- `< 0` = peak shifted toward end

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for curv in [-0.5, -0.2, 0.0, 0.2, 0.5]:
    p = BondWireProfile(start_xy=(20, 100), end_xy=(236, 100),
                        loop_height=40, diameter=2, curvature=curv)
    cps = _bezier_control_points(p)
    curve = _evaluate_bezier(cps)
    ax.plot(curve[:, 0], curve[:, 1], label=f"curvature={curv}")

ax.set_xlim(0, 256)
ax.set_ylim(160, 40)
ax.legend()
ax.set_title("Effect of Curvature Parameter")
ax.set_aspect('equal')
plt.show()